<a href="https://colab.research.google.com/github/Suman-Poola/Random-Forrest-s-p500/blob/main/RNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastai import *
import pandas as pd
from google.colab import userdata
import os
import zipfile
import numpy as np

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

import kaggle


In [2]:
dataset_id = 'darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily'
path = 'ticker_list'
kaggle.api.dataset_download_cli(dataset_id, path = str(path), unzip=True, force=True)


Dataset URL: https://www.kaggle.com/datasets/darkmatternet/s-and-p-500-stocks-25-years-of-data-updated-daily
License(s): CC0-1.0


100%|██████████| 100M/100M [00:06<00:00, 16.0MB/s] 


In [3]:
!ls {path}

sp500_companies.csv  sp500_stocks.csv


In [4]:
tk = pd.read_csv('ticker_list/sp500_companies.csv')
tk.head()

,symbol,company,sector,sub_industry,headquarters,date_added,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1989


In [5]:
import numpy as np
tickers = tk['symbol'].tolist()
tickers[0:25]

['MMM',
 'AOS',
 'ABT',
 'ABBV',
 'ACN',
 'ADBE',
 'AMD',
 'AES',
 'AFL',
 'A',
 'APD',
 'ABNB',
 'AKAM',
 'ALB',
 'ARE',
 'ALGN',
 'ALLE',
 'LNT',
 'ALL',
 'GOOGL',
 'GOOG',
 'MO',
 'AMZN',
 'AMCR',
 'AEE']

In [6]:
len(tickers)

503

In [7]:
tickers[499:503]

['YUM', 'ZBRA', 'ZBH', 'ZTS']

In [8]:
import yfinance as yf
df = yf.download(tickers, period="36mo")
df.head()

/tmp/ipykernel_2944/2136594173.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(tickers, period="36mo")
[*********************100%***********************]  503 of 503 completed


Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-09-12  110.233582  173.947342  134.579880  147.500000  96.199280   
2023-09-13  110.868912  171.885254  136.283859  144.714996  96.471725   
2023-09-14  112.286171  173.394836  138.519806  144.119995  96.434135   
2023-09-15  113.292931  172.674545  137.149368  142.750000  95.579254   
2023-09-18  110.145630  175.595047  138.790253  142.550003  94.884056   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-09-12  74.930809  298.140167  542.210022  167.049286  72.092896  ...   
2023-09-13  74.483887  297.212219  553.559998  168.118088  72.038628  ...   
2023-09-14  75.178032  299.749756  552.159973  171.458145  72.952225  ...   
2023-09-15  75.120987  295.905701  528.890015  169.998093  71.776299  ...   
2023-09-18  76.613892  299.077454  532.419983  171.505859  71.631577  ...   

Price        Volume                                                          \
Ticker           WY     WYNN      XEL       XOM      XYL       XYZ      YUM   
Date                                                                          
2023-09-12  2178900  1292600  4702400  20145800  1582400  10233100  1137500   
2023-09-13  3773300  1260800  6006000  13472800  1740600   5735300  1436500   
2023-09-14  3726200  2066800  5419600  15210700  1906700   8090000  1657800   
2023-09-15  4783700  3658700  9226000  34568400  3608600  11915100  2581200   
2023-09-18  3057800  1268300  4445300  14045300  1854000  11564000  1253800   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-09-12   957100  387600  1675700  
2023-09-13  1160900  407600  1935700  
2023-09-14   996600  444100  2000700  
2023-09-15  2547100  535200  3262600  
2023-09-18  1580900  220000  1228200  

[5 rows x 2515 columns]

In [9]:
df.head()

Price            Close                                                 \
Ticker               A        AAPL        ABBV        ABNB        ABT   
Date                                                                    
2023-09-12  110.233582  173.947342  134.579880  147.500000  96.199280   
2023-09-13  110.868912  171.885254  136.283859  144.714996  96.471725   
2023-09-14  112.286171  173.394836  138.519806  144.119995  96.434135   
2023-09-15  113.292931  172.674545  137.149368  142.750000  95.579254   
2023-09-18  110.145630  175.595047  138.790253  142.550003  94.884056   

Price                                                                 ...  \
Ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
Date                                                                  ...   
2023-09-12  74.930809  298.140167  542.210022  167.049286  72.092896  ...   
2023-09-13  74.483887  297.212219  553.559998  168.118088  72.038628  ...   
2023-09-14  75.178032  299.749756  552.159973  171.458145  72.952225  ...   
2023-09-15  75.120987  295.905701  528.890015  169.998093  71.776299  ...   
2023-09-18  76.613892  299.077454  532.419983  171.505859  71.631577  ...   

Price        Volume                                                          \
Ticker           WY     WYNN      XEL       XOM      XYL       XYZ      YUM   
Date                                                                          
2023-09-12  2178900  1292600  4702400  20145800  1582400  10233100  1137500   
2023-09-13  3773300  1260800  6006000  13472800  1740600   5735300  1436500   
2023-09-14  3726200  2066800  5419600  15210700  1906700   8090000  1657800   
2023-09-15  4783700  3658700  9226000  34568400  3608600  11915100  2581200   
2023-09-18  3057800  1268300  4445300  14045300  1854000  11564000  1253800   

Price                                 
Ticker          ZBH    ZBRA      ZTS  
Date                                  
2023-09-12   957100  387600  1675700  
2023-09-13  1160900  407600  1935700  
2023-09-14   996600  444100  2000700  
2023-09-15  2547100  535200  3262600  
2023-09-18  1580900  220000  1228200  

[5 rows x 2515 columns]

In [10]:
closes = df["Close"]
returns = closes.pct_change() *100

volumes = df['Volume']

cs = closes.stack().rename("Close")
rs = returns.stack().rename("Returns")
vs = volumes.stack().rename("Volume")
fdf = pd.concat([cs,rs,vs], axis = 1).reset_index()

fdf.columns = ['date', 'ticker', 'close', 'return', 'volume']
fdf = fdf.dropna(subset=['return'])

fdf.head(25)



/tmp/ipykernel_2944/4234654860.py:2: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = closes.pct_change() *100


,date,ticker,close,return,volume
495,2023-09-13,A,110.868912,0.576349,2520500.0
496,2023-09-13,AAPL,171.885254,-1.185467,84267900.0
497,2023-09-13,ABBV,136.283859,1.266147,5953700.0
498,2023-09-13,ABNB,144.714996,-1.888138,7893300.0
499,2023-09-13,ABT,96.471725,0.283210,3192500.0
500,2023-09-13,ACGL,74.483887,-0.596447,952400.0
501,2023-09-13,ACN,297.212219,-0.311246,2496200.0
502,2023-09-13,ADBE,553.559998,2.093280,3465200.0
503,2023-09-13,ADI,168.118088,0.639812,2269400.0
504,2023-09-13,ADM,72.038628,-0.075275,2934500.0


fdf.sort_values(by=['ticker', 'date'], inplace=True)
fdf.head(25)

In [11]:
fdf.sort_values(['ticker', 'date'], inplace=True)
fdf.head(25)

,date,ticker,close,return,volume
495,2023-09-13,A,110.868912,0.576349,2520500.0
990,2023-09-14,A,112.286171,1.278320,1815800.0
1485,2023-09-15,A,113.292931,0.896602,6889600.0
1980,2023-09-18,A,110.145630,-2.778020,1818000.0
2475,2023-09-19,A,109.774193,-0.337224,2222400.0
2970,2023-09-20,A,109.930595,0.142477,1744900.0
3465,2023-09-21,A,107.252457,-2.436209,1327900.0
3960,2023-09-22,A,108.894524,1.531030,2050500.0
4455,2023-09-25,A,109.412560,0.475723,987400.0
4950,2023-09-26,A,108.229881,-1.080935,1399800.0


In [12]:
df=fdf

In [13]:
len(df)

375578

In [14]:
df['Lag 1'] = df['close'].shift(1)
df['Lag 2'] = df['close'].shift(2)
df['Lag 3'] = df['close'].shift(3)
df['Lag 5'] = df['close'].shift(5)
df.head(200)

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-09-13,A,110.868912,0.576349,2520500.0,NaN,NaN,NaN,NaN
990,2023-09-14,A,112.286171,1.278320,1815800.0,110.868912,NaN,NaN,NaN
1485,2023-09-15,A,113.292931,0.896602,6889600.0,112.286171,110.868912,NaN,NaN
1980,2023-09-18,A,110.145630,-2.778020,1818000.0,113.292931,112.286171,110.868912,NaN
2475,2023-09-19,A,109.774193,-0.337224,2222400.0,110.145630,113.292931,112.286171,NaN
...,...,...,...,...,...,...,...,...,...
97385,2024-06-24,A,132.735352,1.373356,3339500.0,130.937119,130.426132,132.558441,127.596123
97884,2024-06-25,A,132.352097,-0.288736,2310800.0,132.735352,130.937119,130.426132,129.541748
98383,2024-06-26,A,130.779877,-1.187907,1836800.0,132.352097,132.735352,130.937119,132.558441
98882,2024-06-27,A,128.529648,-1.720623,1898000.0,130.779877,132.352097,132.735352,130.426132


In [15]:
df['Lag 1'] = df.groupby('ticker')['close'].shift(1)
df['Lag 2'] = df.groupby('ticker')['close'].shift(2)
df['Lag 3'] = df.groupby('ticker')['close'].shift(3)
df['Lag 5'] = df.groupby('ticker')['close'].shift(5)
df.head(25)

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5
495,2023-09-13,A,110.868912,0.576349,2520500.0,NaN,NaN,NaN,NaN
990,2023-09-14,A,112.286171,1.278320,1815800.0,110.868912,NaN,NaN,NaN
1485,2023-09-15,A,113.292931,0.896602,6889600.0,112.286171,110.868912,NaN,NaN
1980,2023-09-18,A,110.145630,-2.778020,1818000.0,113.292931,112.286171,110.868912,NaN
2475,2023-09-19,A,109.774193,-0.337224,2222400.0,110.145630,113.292931,112.286171,NaN
2970,2023-09-20,A,109.930595,0.142477,1744900.0,109.774193,110.145630,113.292931,110.868912
3465,2023-09-21,A,107.252457,-2.436209,1327900.0,109.930595,109.774193,110.145630,112.286171
3960,2023-09-22,A,108.894524,1.531030,2050500.0,107.252457,109.930595,109.774193,113.292931
4455,2023-09-25,A,109.412560,0.475723,987400.0,108.894524,107.252457,109.930595,110.145630
4950,2023-09-26,A,108.229881,-1.080935,1399800.0,109.412560,108.894524,107.252457,109.774193


In [16]:
df['SMA 5'] = df['close'].rolling(window=5).mean()
df['SMA 10'] = df['close'].rolling(window=10).mean()
df['SMA 20'] = df['close'].rolling(window=20).mean()
df['SMA 30'] = df['close'].rolling(window=30).mean()
df['SMA 50'] = df['close'].rolling(window=50).mean()

df.head(100)
#df = fdf[fdf['ticker'] == 'AAPL'].copy()
#df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50
495,2023-09-13,A,110.868912,0.576349,2520500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-09-14,A,112.286171,1.278320,1815800.0,110.868912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-15,A,113.292931,0.896602,6889600.0,112.286171,110.868912,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-18,A,110.145630,-2.778020,1818000.0,113.292931,112.286171,110.868912,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-19,A,109.774193,-0.337224,2222400.0,110.145630,113.292931,112.286171,NaN,111.273567,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47600,2024-01-30,A,131.735214,1.091617,1428700.0,130.312698,128.507568,127.418625,129.145233,129.058920,128.689066,128.964735,131.337205,128.178576
48096,2024-01-31,A,127.634438,-3.112893,1101400.0,131.735214,130.312698,128.507568,127.320496,129.121709,128.883315,128.540431,131.126311,128.494529
48592,2024-02-01,A,130.528549,2.267501,862000.0,127.634438,131.735214,130.312698,127.418625,129.743694,129.137408,128.633140,130.987402,128.888735
49088,2024-02-02,A,130.450043,-0.060145,1066300.0,130.528549,127.634438,131.735214,128.507568,130.132188,129.309089,128.729774,130.772097,129.265114


In [17]:
df['volatility 5'] = df['return'].rolling(5).std()
df['volatility 10'] = df['return'].rolling(10).std()
df['volatility 20'] = df['return'].rolling(20).std()
df['volatility 30'] = df['return'].rolling(30).std()
df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,SMA 10,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30
495,2023-09-13,A,110.868912,0.576349,2520500.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
990,2023-09-14,A,112.286171,1.278320,1815800.0,110.868912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1485,2023-09-15,A,113.292931,0.896602,6889600.0,112.286171,110.868912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1980,2023-09-18,A,110.145630,-2.778020,1818000.0,113.292931,112.286171,110.868912,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2475,2023-09-19,A,109.774193,-0.337224,2222400.0,110.145630,113.292931,112.286171,NaN,111.273567,NaN,NaN,NaN,NaN,1.625946,NaN,NaN,NaN


In [56]:
df['Price change per volume'] = np.where(
    df['volume'] != 0,
    1000000 * (df['return'] / df['volume']),
    0
)
df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap,future return
24784,2023-11-21,A,121.366280,8.720838,6085800.0,111.631111,110.818222,111.836784,109.309967,113.382271,...,106.819365,107.875788,3.786238,3.096468,2.641250,2.530027,1.432981,0,7.386109,0.056488
25280,2023-11-22,A,121.434837,0.056488,2630700.0,121.366280,111.631111,110.818222,111.258957,115.417447,...,107.165744,108.087107,3.907064,3.027318,2.556620,2.530109,0.021473,0,3.194586,2.121136
25776,2023-11-24,A,124.010635,2.121136,1226700.0,121.434837,121.366280,111.631111,111.836784,117.852217,...,107.742932,108.321596,3.838424,2.866475,2.570060,2.418457,1.729140,0,1.521238,-1.271526
26272,2023-11-27,A,122.433807,-1.271526,2255400.0,124.010635,121.434837,121.366280,110.818222,120.175334,...,108.216304,108.504414,3.912877,3.003911,2.560043,2.433885,-0.563770,0,2.761372,-0.239979
26768,2023-11-28,A,122.139992,-0.239979,1949800.0,122.433807,124.010635,121.434837,111.631111,122.277110,...,108.647563,108.744301,4.018918,2.950038,2.506809,2.435380,-0.123079,0,2.381486,2.309369


In [50]:
from sklearn.preprocessing import LabelEncoder

df['id'] = LabelEncoder().fit_transform(df['ticker'])
df.head()


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap,future return
24784,2023-11-21,A,121.366280,8.720838,6085800.0,111.631111,110.818222,111.836784,109.309967,113.382271,...,106.819365,107.875788,3.786238,3.096468,2.641250,2.530027,1.432981,0,7.386109,0.056488
25280,2023-11-22,A,121.434837,0.056488,2630700.0,121.366280,111.631111,110.818222,111.258957,115.417447,...,107.165744,108.087107,3.907064,3.027318,2.556620,2.530109,0.021473,0,3.194586,2.121136
25776,2023-11-24,A,124.010635,2.121136,1226700.0,121.434837,121.366280,111.631111,111.836784,117.852217,...,107.742932,108.321596,3.838424,2.866475,2.570060,2.418457,1.729140,0,1.521238,-1.271526
26272,2023-11-27,A,122.433807,-1.271526,2255400.0,124.010635,121.434837,121.366280,110.818222,120.175334,...,108.216304,108.504414,3.912877,3.003911,2.560043,2.433885,-0.563770,0,2.761372,-0.239979
26768,2023-11-28,A,122.139992,-0.239979,1949800.0,122.433807,124.010635,121.434837,111.631111,122.277110,...,108.647563,108.744301,4.018918,2.950038,2.506809,2.435380,-0.123079,0,2.381486,2.309369


In [20]:
df['mk cap'] = df['close'] * df['volume'] / 100000000
df.head()

,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 20,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap
495,2023-09-13,A,110.868912,0.576349,2520500.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.228665,0,2.794451
990,2023-09-14,A,112.286171,1.278320,1815800.0,110.868912,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.703998,0,2.038892
1485,2023-09-15,A,113.292931,0.896602,6889600.0,112.286171,110.868912,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.130138,0,7.805430
1980,2023-09-18,A,110.145630,-2.778020,1818000.0,113.292931,112.286171,110.868912,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.528064,0,2.002448
2475,2023-09-19,A,109.774193,-0.337224,2222400.0,110.145630,113.292931,112.286171,NaN,111.273567,...,NaN,NaN,NaN,1.625946,NaN,NaN,NaN,-0.151738,0,2.439622


In [51]:
df = df.dropna()
#df = df.drop(columns = 'SMA 100')
df.head()


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap,future return
24784,2023-11-21,A,121.366280,8.720838,6085800.0,111.631111,110.818222,111.836784,109.309967,113.382271,...,106.819365,107.875788,3.786238,3.096468,2.641250,2.530027,1.432981,0,7.386109,0.056488
25280,2023-11-22,A,121.434837,0.056488,2630700.0,121.366280,111.631111,110.818222,111.258957,115.417447,...,107.165744,108.087107,3.907064,3.027318,2.556620,2.530109,0.021473,0,3.194586,2.121136
25776,2023-11-24,A,124.010635,2.121136,1226700.0,121.434837,121.366280,111.631111,111.836784,117.852217,...,107.742932,108.321596,3.838424,2.866475,2.570060,2.418457,1.729140,0,1.521238,-1.271526
26272,2023-11-27,A,122.433807,-1.271526,2255400.0,124.010635,121.434837,121.366280,110.818222,120.175334,...,108.216304,108.504414,3.912877,3.003911,2.560043,2.433885,-0.563770,0,2.761372,-0.239979
26768,2023-11-28,A,122.139992,-0.239979,1949800.0,122.433807,124.010635,121.434837,111.631111,122.277110,...,108.647563,108.744301,4.018918,2.950038,2.506809,2.435380,-0.123079,0,2.381486,2.309369


In [22]:
len(df)

372864

In [52]:
df['future return'] = df.groupby('ticker')['return'].shift(-1)
df = df.dropna()
print(len(df))
df.head(len(df))


371858


,date,ticker,close,return,volume,Lag 1,Lag 2,Lag 3,Lag 5,SMA 5,...,SMA 30,SMA 50,volatility 5,volatility 10,volatility 20,volatility 30,Price change per volume,id,mk cap,future return
24784,2023-11-21,A,121.366280,8.720838,6085800.0,111.631111,110.818222,111.836784,109.309967,113.382271,...,106.819365,107.875788,3.786238,3.096468,2.641250,2.530027,1.432981,0,7.386109,0.056488
25280,2023-11-22,A,121.434837,0.056488,2630700.0,121.366280,111.631111,110.818222,111.258957,115.417447,...,107.165744,108.087107,3.907064,3.027318,2.556620,2.530109,0.021473,0,3.194586,2.121136
25776,2023-11-24,A,124.010635,2.121136,1226700.0,121.434837,121.366280,111.631111,111.836784,117.852217,...,107.742932,108.321596,3.838424,2.866475,2.570060,2.418457,1.729140,0,1.521238,-1.271526
26272,2023-11-27,A,122.433807,-1.271526,2255400.0,124.010635,121.434837,121.366280,110.818222,120.175334,...,108.216304,108.504414,3.912877,3.003911,2.560043,2.433885,-0.563770,0,2.761372,-0.239979
26768,2023-11-28,A,122.139992,-0.239979,1949800.0,122.433807,124.010635,121.434837,111.631111,122.277110,...,108.647563,108.744301,4.018918,2.950038,2.506809,2.435380,-0.123079,0,2.381486,2.309369
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373061,2026-09-02,ZTS,76.169998,-1.398061,4474600.0,77.250000,77.099998,77.339996,77.519997,76.575998,...,75.920999,75.477269,2.318983,2.118665,2.708603,2.359121,-0.312444,502,3.408303,0.157546
373564,2026-09-03,ZTS,76.290001,0.157546,3129100.0,76.169998,77.250000,77.099998,75.019997,76.829999,...,75.978333,75.450492,1.663900,1.983658,2.563710,2.357949,0.050349,502,2.387190,-0.629183
374067,2026-09-04,ZTS,75.809998,-0.629183,4169500.0,76.290001,76.169998,77.250000,77.339996,76.523999,...,75.993666,75.421066,0.655935,1.582414,2.159469,2.354682,-0.150901,502,3.160898,-2.967946
374570,2026-09-08,ZTS,73.559998,-2.967946,5657000.0,75.809998,76.290001,76.169998,77.099998,75.815999,...,75.881666,75.381001,1.314360,1.802734,2.170516,2.384466,-0.524650,502,4.161289,-0.339859


In [53]:
features = ['close', 'return', 'Lag 1', 'Lag 2', 'Lag 3', 'Lag 5', 'volatility 5', 'volatility 10', 'volatility 20', 'volatility 30','SMA 10' ,'SMA 20','SMA 30','SMA 50','volume','Price change per volume','id','mk cap']
target = 'future return'
x = df[features]
y = df['future return']

In [25]:
split_index = int(0.8 * len(df))

x_train, x_test = x.iloc[:split_index], x.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

In [26]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [27]:
device = torch.device("cpu")


In [39]:
def make_sequences(window, ticker_col='ticker', date_col='date'):
    X_list, y_list = [], []

    for ticker, group in df.groupby(target):
        group = group.sort_values(date_col)
        feat_arr = group[features].values
        target_arr = group[ticker_col].values

        n_days = len(group)
        for i in range(n_days - window):
            X_list.append(feat_arr[i : i + window])
            y_list.append(target_arr[i + window])
    return np.array(X_list), np.array(y_list)

In [58]:
X_seq, y_seq = make_sequences(10)
X_seq.shape
y_seq.shaped

(1308,)

In [59]:
train_pct = 0.8
X_train, y_train = [], []
X_test, y_test = [], []
for ticker, group in df.groupby('ticker'):
    group = group.sort_values('date')
    feat_arr = group[features].values
    target_arr = group[target].values

    n_days = len(group)


    ticker_X, ticker_y = [], []
    for i in range(n_days - 10 ): #10 is the window, be sure to change it if you change the window
         ticker_X.append(feat_arr[i : i + 10])
         ticker_y.append(target_arr[i + 10])

    n_seqs = len(ticker_X)
    split_idx = int(train_pct * n_seqs)

    X_train.extend(ticker_X[:split_idx])
    y_train.extend(ticker_y[:split_idx])
    X_test.extend(ticker_X[split_idx:])
    y_test.extend(ticker_y[split_idx:])

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)

train: (293459, 10, 18) (293459,)
test:  (73369, 10, 18) (73369,)


In [60]:
print(x_train.shape)
print(y_train.shape)
print(x_test.shape)
print(y_test.shape)

(297888, 18)
(293459,)
(74473, 18)
(73369,)


In [61]:
print(x_train.shape[0] + x_test.shape[0])
print(len(tickers)*(750-10) )

372361
372220


In [62]:
import numpy as np
for i, f in enumerate(features):
    n_inf = np.isinf(X_train[:, :, i]).sum()
    if n_inf > 0:
        print(f, n_inf)

In [63]:
n_features = X_train.shape[2]

scaler = StandardScaler()
scaler.fit(X_train.reshape(-1, n_features))

x_train_scaled = scaler.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
x_test_scaled = scaler.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

print(x_train_scaled.shape)
x_test_scaled.shape

(293459, 10, 18)


(73369, 10, 18)

In [66]:
from torch.utils.data import Dataset, DataLoader

class StockSequenceDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_ds = StockSequenceDataset(x_train_scaled, y_train)
test_ds = StockSequenceDataset(x_test_scaled, y_test)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=256, shuffle=False)

xb, yb = next(iter(train_loader))
print(xb.shape)
yb.shape

torch.Size([256, 10, 18])


torch.Size([256, 1])

In [70]:
class StockLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.rnn = nn.LSTM(input_size=input_size, hidden_size=hidden_size,
                            num_layers=num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, (h_n, c_n) = self.rnn(x)
        last_step = out[:, -1, :]
        return self.fc(last_step)

model = StockLSTM(input_size=18, hidden_size=64, num_layers=1).to(device)

In [73]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

n_epochs = 20
train_losses, test_losses = [], []

for epoch in range(n_epochs):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        pred = model(xb)
        loss = criterion(pred, yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * xb.size(0)
    train_loss = running_loss / len(train_ds)

    model.eval()
    running_test_loss = 0.0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            running_test_loss += loss.item() * xb.size(0)
    test_loss = running_test_loss / len(test_ds)

    train_losses.append(train_loss)
    test_losses.append(test_loss)
    print(f"Epoch {epoch+1}")
    print(train_loss)
    print(test_loss)

Epoch 1
4.1933
 6.6803
Epoch 2
4.1705
 6.7403
Epoch 3
4.1509
 6.6995
Epoch 4
4.1328
 6.7512
Epoch 5
4.1121
 6.7811
Epoch 6
4.0921
 6.8134
Epoch 7
4.0726
 6.8231
Epoch 8
4.0522
 6.8553
Epoch 9
4.0316
 6.8745
Epoch 10
4.0103
 6.8714
Epoch 11
3.9872
 6.9714
Epoch 12
3.9699
 6.9791
Epoch 13
3.9524
 6.9654
Epoch 14
3.9302
 7.0842
Epoch 15
3.9103
 7.0654
Epoch 16
3.8938
 7.1299
Epoch 17
3.8738
 7.1155
Epoch 18
3.8562
 7.1496
Epoch 19
3.8346
 7.2508
Epoch 20
3.8181
 7.1509
